# 🔧 Input Preprocessing — Feature Ordering & Type Conversion

> **Module 3 of 8** — The Cervical Cancer Prediction Service

---

## 🎯 What This Module Does

This notebook demonstrates how the API **converts validated JSON input into a model-ready NumPy array**. The preprocessor ensures features are in the **exact same order** as the columns used during model training — a critical requirement for correct predictions.

## 🔄 The Data Pipeline

```
JSON Request ──→ Pydantic Validation ──→ Dict (by_alias) ──→ Preprocessor ──→ NumPy Array (1, 35) ──→ Model
```

## ⚠️ Why Feature Ordering Is Critical

| Scenario | Result |
|----------|--------|
| Features in correct order | ✅ Model predicts correctly |
| Features shuffled | ❌ Model produces garbage predictions |
| Missing features | ❌ `ValueError` — clear error message |
| Extra features | ❌ `ValueError` — clear error message |

The preprocessor reads the **feature_columns.json** file (35 features in training order) and builds the vector by looking up each column name in the input dict. Missing keys default to `0.0` (safe for this dataset).

---

## 🧠 Skills Demonstrated

- ✅ **NumPy** — efficient array manipulation
- ✅ **Data Pipelines** — clean separation of concerns
- ✅ **Reproducibility** — deterministic feature ordering
- ✅ **Error Handling** — clear messages for missing/invalid config

In [ ]:
# ── Add the api directory to sys.path so 'app' module can be found ──
import sys
from pathlib import Path

_candidates = [
    Path.cwd() / "cervical-cancer-mlops" / "api",
    Path.cwd().parent / "api",
    Path.cwd() / "api",
]
api_dir = next((p for p in _candidates if (p / "app").exists()), None)
if api_dir is None:
    raise RuntimeError(
        "Could not locate the 'api/app' package. "
        "Expected it under one of: " + ", ".join(map(str, _candidates))
    )
if str(api_dir) not in sys.path:
    sys.path.insert(0, str(api_dir))
print(f"✅ Added {api_dir} to sys.path")


In [ ]:
# ── Import the preprocessor ──
from app.preprocessor import load_feature_columns, preprocess

# Load the feature columns
feature_columns = load_feature_columns()
print(f"✅ Loaded {len(feature_columns)} feature columns")
print(f"\nFeature order (first 10):")
for i, col in enumerate(feature_columns[:10], 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# ── Demonstrate preprocessing a sample patient ──
sample_patient = {
    "Age": 25,
    "Number of sexual partners": 1,
    "First sexual intercourse": 22,
    "Num of pregnancies": 0,
    "Smokes": 0,
    "Smokes (years)": 0,
    "Smokes (packs/year)": 0.0,
    "Hormonal Contraceptives": 0,
    "Hormonal Contraceptives (years)": 0.0,
    "IUD": 0,
    "IUD (years)": 0.0,
    "STDs": 0,
    "STDs (number)": 0,
    "STDs:condylomatosis": 0,
    "STDs:cervical condylomatosis": 0,
    "STDs:vaginal condylomatosis": 0,
    "STDs:vulvo-perineal condylomatosis": 0,
    "STDs:syphilis": 0,
    "STDs:pelvic inflammatory disease": 0,
    "STDs:genital herpes": 0,
    "STDs:molluscum contagiosum": 0,
    "STDs:AIDS": 0,
    "STDs:HIV": 0,
    "STDs:Hepatitis B": 0,
    "STDs:HPV": 0,
    "STDs: Number of diagnosis": 0,
    "STDs: Time since first diagnosis": 0.0,
    "STDs: Time since last diagnosis": 0.0,
    "Dx:Cancer": 0,
    "Dx:CIN": 0,
    "Dx:HPV": 0,
    "Dx": 0,
    "Hinselmann": 0,
    "Schiller": 0,
    "Citology": 0
}

features_array = preprocess(sample_patient)
print(f"✅ Preprocessed successfully!")
print(f"   Shape: {features_array.shape}")
print(f"   Dtype: {features_array.dtype}")
print(f"   Values: {features_array[0][:5]}... (first 5 of {features_array.shape[1]})")

In [ ]:
# ── Demonstrate missing features default to 0.0 ──
# Remove a few keys to show they default to 0.0
partial_patient = dict(sample_patient)
del partial_patient["Age"]
del partial_patient["Smokes"]

features_array = preprocess(partial_patient)
print("✅ Missing 'Age' and 'Smokes' defaulted to 0.0:")
print(f"   Age value:    {features_array[0][0]}")
print(f"   Smokes value: {features_array[0][4]}")

---

## ✅ Summary

The preprocessor guarantees that every prediction request is converted into a **correctly-ordered, model-ready NumPy array**. This is a critical step that prevents silent prediction errors caused by feature misalignment.

**Next:** [Module 4 — MLflow Model Registry Integration](api_04_model_loader.ipynb)